In [12]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [13]:
# load datasets
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")
test_data = pd.read_csv("samsum-test.csv")

In [14]:
train_data.head(5)

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [15]:
# the data is too big for the training so we are not going to train all the data
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state = 42).reset_index(drop=True)

In [16]:
train_data.shape

(4000, 3)

In [17]:
val_data.shape

(500, 3)

#### Data Preprocessing

In [18]:
import re
def clean_data(text):
  text = re.sub(r"\r\n", " ", text)           # lines
  text = re.sub(r"s+", " ", text)             # spaces
  text = re.sub(r"<.*?>", " ", text)          # html tags
  text = text.strip().lower()                 # spaces and lowercase
  return text

In [19]:
# apply the function
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
print(train_data['dialogue'])

0       violet: hi! i came acro  thi  au tin'  article...
1       pat: so doe  anyone know when the  tream i  go...
2       jane:   jane: whaddya think?  shona: thi  ur t...
3       adam: do u have a map of pari ? tom: ye , why?...
4       frank: hi, how'  the family? mike: great! sam'...
                              ...                        
3995    barry: hello buddy michael: hey barry: do you ...
3996    karen: hey li a. lari a and me have recently m...
3997    mile : hey, guy , i'm  o  orry, but i mi ed th...
3998    emma: did you fini h the book i gave you? liam...
3999    jenna: dude , were we  uppo ed to read the who...
Name: dialogue, Length: 4000, dtype: object


In [20]:
# clean for the summary col and also for the val_data
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

#### Tokenization

In [21]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
from numpy import True_
# raw data => tokenized inputs for fine-tuning
def tokenize(data):
  inputs = tokenizer(data['dialogue'], padding = 'max_length', max_length = 512, truncation=True)
  targets = tokenizer(data['summary'], padding = 'max_length', max_length = 150, truncation=True)

  inputs['labels'] = targets['input_ids']         # token ids => add to input as labels
  return inputs

In [23]:
train_data = train_data.apply(tokenize, axis=1).tolist()
val_data = val_data.apply(tokenize, axis=1).tolist()

In [24]:
train_data[1]

{'input_ids': [6234, 10, 78, 103, 15, 1321, 214, 116, 8, 3, 929, 265, 3, 23, 352, 12, 1837, 58, 16585, 10, 12050, 6, 150, 6, 68, 133, 310, 114, 12, 5, 3, 1050, 2494, 10, 3, 23, 278, 31, 17, 317, 3, 23, 31, 26, 36, 1413, 15, 3, 1054, 16, 3, 7436, 3, 5, 6234, 10, 3, 63, 58, 3, 1050, 2494, 10, 2492, 66, 8, 1717, 11, 3224, 3640, 143, 140, 1227, 19974, 5, 16585, 10, 78, 25, 31, 60, 3, 32, 7569, 58, 6234, 10, 3, 75, 31, 2157, 55, 3, 7, 52, 3, 120, 58, 3, 1050, 2494, 10, 3, 63, 413, 5, 141, 8, 183, 15, 589, 16, 565, 3, 23, 8036, 3, 9, 861, 5, 16585, 10, 2087, 34, 31, 97, 12, 483, 34, 58, 6234, 10, 17945, 55, 428, 34, 3, 9, 653, 55, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [25]:
len(train_data[0]['input_ids'])

512

In [26]:
len(train_data[0]['labels'])

150

In [27]:
# input ids - dialogue => token ids
# 1 => EOS, 0 => padding
# attention mask
# labels -target => summary token

In [28]:
print(type(train_data))
print(type(val_data))

<class 'list'>
<class 'list'>


#### Working with Model

In [29]:
# NLP => generation task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [30]:

# train the model selecting device
import torch

if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print(f"device: ", device)

device:  cuda


In [31]:
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [33]:
# Training arguments
training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
)

In [34]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_data,
    eval_dataset = val_data
)

In [35]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.436326,0.516488
2,0.528022,0.469247
3,0.488924,0.454098
4,0.471714,0.447558
5,0.460139,0.443953
6,0.455427,0.442476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9734252370198567, metrics={'train_runtime': 1252.7265, 'train_samples_per_second': 19.158, 'train_steps_per_second': 2.395, 'total_flos': 3248203235328000.0, 'train_loss': 0.9734252370198567, 'epoch': 6.0})

In [36]:
# model load => fine tune => save the model
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [38]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## test the model

In [39]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue)       # clean the dialogue

  # tokenize
  inputs = tokenizer(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"
  )

  # generate the summary => token ids
  model.to(device)              # ensure that the model and the data on the same device
  targets = model.generate(
      input_ids = inputs['input_ids'],
      attention_mask = inputs["attention_mask"],
      max_length = 150,
      num_beams = 4,
      early_stopping = True
  )

  # token ids convert to summary => decoding
  summary = tokenizer.decode(targets[0], skip_special_tokens=True)     # EOS, SEP
  return summary

